# Learning to use Quantipy

In [ ]:
# utils/qp_helpers.py
from __future__ import annotations  # must be first

from dataclasses import dataclass
from typing import Dict, Mapping, Optional
import pandas as pd
from quantipy.core.dataset import DataSet


@dataclass(frozen=True)
class CrosstabSpec:
    """Configuration for a single crosstab."""
    row: str
    col: str
    weight: Optional[str] = None
    row_labels: Optional[Mapping[int, str]] = None
    col_labels: Optional[Mapping[int, str]] = None
    sheet_name: str = "table"


def encode_categoricals(
    df: pd.DataFrame,
    maps: Mapping[str, Mapping[object, int]],
    ensure_int: bool = True,
) -> pd.DataFrame:
    """Map string categories to integer codes for Quantipy.

    Examples
    --------
    >>> import pandas as pd
    >>> df = pd.DataFrame({'gender': ['F','M','F'], 'q1':[1,2,1]})
    >>> maps = {'gender': {'F':1, 'M':2}}
    >>> out = encode_categoricals(df, maps)
    >>> set(out['gender'].unique()) == {1,2}
    True
    """
    out = df.copy()
    for col, mapping in maps.items():
        out[col] = out[col].map(mapping)
        if ensure_int:
            out[col] = out[col].astype("int64")
    return out


def build_dataset(df: pd.DataFrame, name: str = "ds") -> DataSet:
    """Construct a Quantipy DataSet from a pandas DataFrame."""
    ds = DataSet(name=name)
    ds.from_components(df)
    return ds


def run_crosstab(ds: DataSet, spec: CrosstabSpec) -> pd.DataFrame:
    """Run a crosstab and return a labeled pandas DataFrame."""
    tab = ds.crosstab(spec.row, spec.col, w=spec.weight)  # returns DataFrame in your version
    if spec.row_labels:
        tab = tab.rename(index=lambda x: spec.row_labels.get(x, x))
    if spec.col_labels:
        tab = tab.rename(columns=lambda x: spec.col_labels.get(x, x))
    return tab


def export_tables_xlsx(tables: Mapping[str, pd.DataFrame], path: str) -> str:
    """Export multiple tables to an Excel workbook."""
    with pd.ExcelWriter(path, engine="xlsxwriter") as xw:
        for sheet, df in tables.items():
            df.to_excel(xw, sheet_name=sheet)
    return path

SyntaxError: from __future__ imports must occur at the beginning of the file (<ipython-input-9-4c6452b579a8>, line 5)

In [5]:
import pandas as pd
from quantipy.core.dataset import DataSet

# Raw data
df = pd.DataFrame([
    {"gender": "F", "q1": 1},
    {"gender": "M", "q1": 2},
    {"gender": "F", "q1": 1},
])

# Encode gender into numeric categories
map_gender = {"F": 1, "M": 2}
df["gender"] = df["gender"].map(map_gender).astype("int64")
df["q1"] = df["q1"].astype("int64")

# Build DataSet
ds = DataSet(name="demo")
ds.from_components(df)

# Crosstab
tab = ds.crosstab("gender", "q1")
print(type(tab))   # should say <class 'pandas.core.frame.DataFrame'>
print(tab)

Inferring meta data from pd.DataFrame.columns (2)...
Converted 2 columns!
<class 'pandas.core.frame.DataFrame'>
Question        q1.      
Values             1    2
Question Values          
gender.  Base    2.0  1.0
         1       2.0  0.0
         2       0.0  1.0


In [6]:
import pandas as pd
from quantipy.core.dataset import DataSet

# ---------------------------
# 1) Sample data
# ---------------------------
raw = pd.DataFrame([
    {"gender": "F", "q1": 1},
    {"gender": "M", "q1": 2},
    {"gender": "F", "q1": 1},
])

# Quantipy wants categorical codes (ints)
gender_map = {"F": 1, "M": 2}
q1_labels   = {1: "Choice 1", 2: "Choice 2"}  # optional, for nicer column labels

df = raw.copy()
df["gender"] = df["gender"].map(gender_map).astype("int64")
df["q1"]     = df["q1"].astype("int64")

# ---------------------------
# 2) Build DataSet and crosstab
# ---------------------------
ds = DataSet(name="demo")
ds.from_components(df)

tab = ds.crosstab("gender", "q1")  # returns a pandas DataFrame in your version

# ---------------------------
# 3) Replace codes with labels for display
# ---------------------------
index_labels = {1: "F", 2: "M"}
col_labels   = q1_labels              # or {1:"1", 2:"2"} if you prefer raw

tab_labeled = tab.copy()
# Row index in your output has a multiindex-like format; reindex safely:
# Keep "Base" row as-is, remap only numeric categories if present.
tab_labeled = tab_labeled.rename(index=lambda x: index_labels.get(x, x))
tab_labeled = tab_labeled.rename(columns=lambda x: col_labels.get(x, x))

print("Crosstab (labeled):")
print(tab_labeled)

# ---------------------------
# 4) Weighted example (optional)
# ---------------------------
# If you have a weight column (e.g., 'w'): ensure it's numeric and in df.
# df["w"] = [1.2, 0.8, 1.0]  # example
# ds = DataSet(name="demo_w"); ds.from_components(df)
# tab_w = ds.crosstab("gender", "q1", w="w")  # returns DataFrame
# tab_w = tab_w.rename(index=lambda x: index_labels.get(x, x)).rename(columns=lambda x: col_labels.get(x, x))
# print(tab_w)

# ---------------------------
# 5) Export to Excel
# ---------------------------
out_path = "quantipy_demo_crosstab.xlsx"
with pd.ExcelWriter(out_path, engine="xlsxwriter") as xw:
    tab_labeled.to_excel(xw, sheet_name="ct_gender_q1")
print("Saved:", out_path)

Inferring meta data from pd.DataFrame.columns (2)...
Converted 2 columns!
Crosstab (labeled):
Question        q1.      
Values             1    2
Question Values          
gender.  Base    2.0  1.0
         1       2.0  0.0
         2       0.0  1.0
Saved: quantipy_demo_crosstab.xlsx
